In [ ]:
from spark_session import spark

In [ ]:
spark.conf.get("spark.sql.parquet.compression.codec"), spark.conf.get("spark.sql.files.maxRecordsPerFile")

In [ ]:
# adequate memory + cap required for repartitioning
# spark.sparkContext.getConf().get("spark.driver.memory")
# spark.sparkContext.getConf().getAll()

In [ ]:
spark.sql("""
SELECT *
FROM nessie.tpch.nation
""").show()

In [ ]:
spark.sql("""
SELECT
    count(*) AS files,
    sum(file_size_in_bytes) / 1024 / 1024 / 1024 AS gb,
    avg(file_size_in_bytes) / 1024 / 1024 AS avg_mb
FROM nessie.tpch.lineitem.files;
""").show(truncate=False)

In [ ]:
# tables
tables = [
    "customer",
    "lineitem",
    "nation",
    "orders",
    "part",
    "partsupp",
    "region",
    "supplier",
]
data_dir_path = "/home/tushar/lake-forge/data_generator/data/data_sf_10"

In [ ]:
total_data_size_bytes = 0
for i, table in enumerate(tables, start=1):
    print(f"#{i}/{len(tables)} {table} table")

    src_count = spark.read.parquet(f"{data_dir_path}/{table}.parquet").count()
    print(f"Records count in source parquet file: {src_count}")

    iceberg_count = spark.sql(f"select count(*) from nessie.tpch.{table}").collect()[0][0]
    print(f"Records count in iceberg table: {iceberg_count}")

    size_bytes = spark.sql(f"""
        SELECT SUM(file_size_in_bytes)
        FROM nessie.tpch.{table}.files
    """).collect()[0][0]
    total_data_size_bytes += size_bytes
    print(f"Total data size in iceberg: {size_bytes/1024/1024:.2f} MB ({size_bytes} bytes)")
    print("-"*50)

print(f"Total data size in iceberg for all tables: {total_data_size_bytes/1024/1024:.2f} MB ({total_data_size_bytes} bytes)")


In [ ]:
# snapshots
print("Snapshots")
spark.sql("SELECT * FROM nessie.tpch.customer.snapshots").show()

# files
print("Files")
spark.sql("select * from nessie.tpch.customer.files").show()

# history
print("History")
spark.sql("select * from nessie.tpch.customer.history").show()

## Schema Evolution and Time Travel

In [ ]:
spark.sql("select * from nessie.tpch.customer").show(5)

In [ ]:
spark.sql("alter table nessie.tpch.customer add column ingestion_ts timestamp")

In [ ]:
from pyspark.sql.functions import current_timestamp

spark.table("nessie.tpch.customer").withColumn("ingestion_ts", current_timestamp()).writeTo("nessie.tpch.customer").overwritePartitions()

In [ ]:
spark.sql("select * from nessie.tpch.customer").show(5)

In [ ]:
spark.sql("select * from nessie.tpch.customer.snapshots").show(5, truncate=False)

In [ ]:
snapshot_ids = spark.sql("select snapshot_id from nessie.tpch.customer.snapshots").collect()
snapshot_id_0, snapshot_id_1 = snapshot_ids[0][0], snapshot_ids[1][0]

# customer table right now
print(f"Customer table right now (snapshot_id: {snapshot_id_1}):")
spark.sql("select * from nessie.tpch.customer").show(5)

# customer table before adding timestamp column and overwriting partitions
print(f"Customer table before adding timestamp column (snapshot_id: {snapshot_id_0}):")
spark.sql(f"select * from nessie.tpch.customer version as of {snapshot_id_0}").show(5)


## Branching in Nessie

In [ ]:
# list branches
spark.conf.get("spark.sql.catalog.nessie.uri")

# print("Branches:")
spark.sql("LIST REFERENCES IN nessie").show(truncate=False)

spark.sql("CREATE BRANCH if not exists experiment IN nessie").show()

In [ ]:
spark.sql("LIST REFERENCES IN nessie").show(truncate=False)

In [ ]:
spark.sql("USE REFERENCE experiment IN nessie").show()
spark.sql("create table if not exists nessie.tpch.users (id int, name string)").show()
spark.sql("insert into nessie.tpch.users values (1, 'Alice')")
spark.sql("select * from nessie.tpch.users").show()
spark.sql("show tables in nessie.tpch").show()

In [ ]:
spark.sql("USE REFERENCE main IN nessie").show()

In [ ]:
spark.sql("show tables in nessie.tpch").show()

## Repartitioning

In [ ]:
spark.sql("""
ALTER TABLE nessie.tpch.lineitem ADD PARTITION FIELD months(l_shipdate)
"""
)

In [ ]:
# check
spark.sql("DESCRIBE EXTENDED nessie.tpch.lineitem").show(truncate=False)
spark.sql("""
SELECT *
FROM nessie.tpch.lineitem.partitions
""").show(truncate=False)

In [ ]:
spark.sql("""
    CALL nessie.system.rewrite_data_files(
        table => 'tpch.lineitem'
    )
""")
# DataFrame[rewritten_data_files_count: int, added_data_files_count: int, rewritten_bytes_count: bigint, failed_data_files_count: int, removed_delete_files_count: int]

# rewritten_data_files_count 203
# added_data_files_count 84
# rewritten_bytes_count 16214863653
# failed_data_files_count 0
# removed_delete_files_count 0



In [ ]:
spark.sql("""
SELECT *
FROM nessie.tpch.lineitem.partitions
ORDER BY record_count DESC;
""").show(truncate=False)

In [ ]:
# spark.sql("""
# EXPLAIN EXTENDED
# SELECT *
# FROM nessie.tpch.lineitem
# WHERE l_shipdate = DATE '1995-03-15';
# """).show(truncate=False)

spark.sql("""
SELECT *
FROM nessie.tpch.lineitem
WHERE l_shipdate < DATE '1995-03-15';
""").show(truncate=False)

# spark.sql("""
# SELECT min(c_acctbal), max(c_acctbal)
# FROM nessie.tpch.customer
# """).show(truncate=False)

# spark.sql("""
# SELECT *
# FROM nessie.tpch.customer
# where c_acctbal < 1000
# """).show(truncate=False)